# Data Exploration
Load the dataset, examine its structure (modulations, SNRs), and visualize IQ samples and spectrograms.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import sys
sys.path.insert(0, '..')

# Check data directory structure
data_dir = Path('../data')
print("Data directory contents:")
for item in data_dir.glob('*'):
    print(f"  {item.name}: {item.is_dir()}")

## Load Sample Dataset
For this exploration, we'll create a synthetic dataset mimicking RadioML format (IQ samples with modulation labels and SNR).

In [ ]:
# Create synthetic dataset for exploration
from src.data_preprocessing.load_data import create_synthetic_dataset, load_dataset

# Generate synthetic data
synthetic_dir = data_dir / 'synthetic'
synthetic_dir.mkdir(exist_ok=True)
print("Creating synthetic dataset...")
create_synthetic_dataset(output_dir=str(synthetic_dir), num_samples=500, seed=42)
print("Synthetic dataset created.")

# Load the dataset
dataset = load_dataset(str(synthetic_dir))
print(f"\nDataset loaded: {len(dataset)} samples")
print(f"Keys: {dataset[0].keys()}")

## Examine Dataset Structure

In [ ]:
# Extract structure info
modulations = []
snrs = []
sample_iq = []

for sample in dataset[:100]:
    modulations.append(sample['modulation'])
    snrs.append(sample['snr'])
    sample_iq.append(sample['iq_data'])

modulations_unique = list(set(modulations))
print(f"Unique modulations: {modulations_unique}")
print(f"Number of modulations: {len(modulations_unique)}")
print(f"\nSNR range: {min(snrs):.1f} dB to {max(snrs):.1f} dB")
print(f"SNR values: {sorted(set(snrs))}")

# Modulation distribution
mod_counts = pd.Series(modulations).value_counts()
print(f"\nModulation distribution:\n{mod_counts}")

## Visualize IQ Samples

In [ ]:
# Plot IQ constellation for different modulations
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for idx, mod in enumerate(modulations_unique[:6]):
    iq_sample = next(s['iq_data'] for s in dataset if s['modulation'] == mod)
    axes[idx].scatter(iq_sample.real, iq_sample.imag, alpha=0.5, s=10)
    axes[idx].set_title(f'{mod}')
    axes[idx].set_xlabel('I')
    axes[idx].set_ylabel('Q')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].axis('equal')

plt.tight_layout()
plt.savefig('../data/iq_constellations.png', dpi=100, bbox_inches='tight')
plt.show()
print("IQ constellation plot saved.")

## Convert IQ to Spectrograms

In [ ]:
from src.data_preprocessing.create_spectrograms import iq_to_spectrogram

# Convert first 5 samples to spectrograms
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
axes = axes.flatten()

for idx in range(min(6, len(dataset))):
    sample = dataset[idx]
    iq_data = sample['iq_data']
    mod = sample['modulation']
    snr = sample['snr']
    
    # Convert to spectrogram
    spec = iq_to_spectrogram(iq_data, n_fft=256)
    
    # Plot
    im = axes[idx].imshow(spec, aspect='auto', origin='lower', cmap='viridis')
    axes[idx].set_title(f'{mod} @ {snr}dB')
    axes[idx].set_xlabel('Time')
    axes[idx].set_ylabel('Frequency')
    plt.colorbar(im, ax=axes[idx])

plt.tight_layout()
plt.savefig('../data/spectrograms_sample.png', dpi=100, bbox_inches='tight')
plt.show()
print("Spectrogram plot saved.")

## Dataset Statistics Summary

In [ ]:
# Summary statistics
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Total samples: {len(dataset)}")
print(f"IQ sample length: {len(dataset[0]['iq_data'])}")
print(f"Number of modulation classes: {len(modulations_unique)}")
print(f"SNR range: {min(snrs):.1f} - {max(snrs):.1f} dB")
print(f"\nModulation classes:")
for mod in sorted(modulations_unique):
    count = sum(1 for s in dataset if s['modulation'] == mod)
    print(f"  {mod}: {count} samples")
print("=" * 50)